In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import json

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    make_scorer,
    matthews_corrcoef,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

import sys
sys.path.append("../../utils/")

from utils import *

import time

In [ ]:
# ===== RUTAS =====
PROJECT_ROOT = Path.cwd().resolve().parents[2]

ESTRATEGIA_DE_REBALANCEO = "NearMiss_SMOTE"
MODELO = "mlp"

NOMBRE_EXPERIMENTO = f"BCCC17__split__v1__{ESTRATEGIA_DE_REBALANCEO}_pca4_{MODELO}__v1"
CARPETA_DATASET = "BCCC17__split__v1"

NOMBRE_DATASET_TRAIN = f"{CARPETA_DATASET}__train.csv"
NOMBRE_DATASET_TEST = f"{CARPETA_DATASET}__test.csv"

RUTA_DATASET = PROJECT_ROOT / "02_datasets" / "processed" / CARPETA_DATASET
RUTA_RESULTADOS = PROJECT_ROOT / "04_experimentos" / "logs" / "resultados" / NOMBRE_EXPERIMENTO

NOMBRE_RESULTADOS_CV_CSV = f"{NOMBRE_EXPERIMENTO}__folds.csv"
NOMBRE_RESULTADOS_CV_JSON = f"{NOMBRE_EXPERIMENTO}__summary_cv.json"
NOMBRE_RESULTADOS_TEST_JSON = f"{NOMBRE_EXPERIMENTO}__summary_test.json"
NOMBRE_RESULTADOS_TEST_CSV = f"{NOMBRE_EXPERIMENTO}__metricas_test.csv"
NOMBRE_RESULTADOS_TEST_CM_CSV = f"{NOMBRE_EXPERIMENTO}__confusion_matrix_test.csv"

# ===== PARÁMETROS =====
LABEL_COL = "LABEL"

N_SPLITS = 5
SHUFFLE = True
RANDOM_STATE = 42

# ===== CONFIG MLP =====
MLP_HIDDEN_LAYER_SIZES = (100,)
MLP_ACTIVATION = "relu"
MLP_SOLVER = "adam"
MLP_ALPHA = 0.0001
MLP_BATCH_SIZE = "auto"
MLP_LEARNING_RATE = "adaptive"
MLP_LEARNING_RATE_INIT = 0.001
MLP_MAX_ITER = 300
MLP_EARLY_STOPPING = True
MLP_VALIDATION_FRACTION = 0.1
MLP_N_ITER_NO_CHANGE = 10
MLP_VERBOSE = False

# ===== CONFIG PCA =====
N_COMPONENTS_PCA = 46

# ===== CONFIG REBALANCEO DENTRO DEL CV =====
TARGET_N = 10000
NEARMISS_VERSION = 1
SMOTE_K_NEIGHBORS = 5
ENN_N_NEIGHBORS = 3

In [ ]:
RUTA_RESULTADOS.mkdir(parents=True, exist_ok=True)

print("Ruta dataset train:")
print((RUTA_DATASET / NOMBRE_DATASET_TRAIN).resolve())
print()

print("Ruta dataset test:")
print((RUTA_DATASET / NOMBRE_DATASET_TEST).resolve())
print()

print("Ruta resultados:")
print(RUTA_RESULTADOS.resolve())

In [ ]:
df_train = cargar_dataset(
    nombre_dataset=NOMBRE_DATASET_TRAIN,
    ruta_base=RUTA_DATASET
)

print("Forma del dataset train:")
print(df_train.shape)

df_train.head()

In [ ]:
if LABEL_COL not in df_train.columns:
    raise ValueError(f"No se encontró la columna {LABEL_COL} en train")

print("Última columna train:", df_train.columns[-1])
print("Tipo de LABEL train:", df_train[LABEL_COL].dtype)
print()

print("Distribución de clases en train:")
display(df_train[LABEL_COL].value_counts(dropna=False).to_frame("count"))

In [ ]:
X_train = df_train.drop(columns=[LABEL_COL]).copy()
y_train = df_train[LABEL_COL].copy()

print("Shape X_train:", X_train.shape)
print("Shape y_train:", y_train.shape)

In [ ]:
columnas_no_numericas_train = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

print("Columnas no numéricas en X_train:")
print(columnas_no_numericas_train)

if len(columnas_no_numericas_train) > 0:
    raise ValueError("Hay columnas no numéricas en X_train. Revísalas antes de seguir.")

In [ ]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=N_COMPONENTS_PCA)),
    ("mlp", MLPClassifier(
        hidden_layer_sizes=MLP_HIDDEN_LAYER_SIZES,
        activation=MLP_ACTIVATION,
        solver=MLP_SOLVER,
        alpha=MLP_ALPHA,
        batch_size=MLP_BATCH_SIZE,
        learning_rate=MLP_LEARNING_RATE,
        learning_rate_init=MLP_LEARNING_RATE_INIT,
        max_iter=MLP_MAX_ITER,
        early_stopping=MLP_EARLY_STOPPING,
        validation_fraction=MLP_VALIDATION_FRACTION,
        n_iter_no_change=MLP_N_ITER_NO_CHANGE,
        verbose=MLP_VERBOSE,
        random_state=RANDOM_STATE
    ))
])

pipeline

In [ ]:
cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=SHUFFLE,
    random_state=RANDOM_STATE
)

cv

In [ ]:
labels_globales = np.array(sorted(y_train.unique()))

resultados_folds = []

for fold, (train_idx, val_idx) in enumerate(cv.split(X_train, y_train), start=1):

    print("=" * 80)
    print(f"FOLD {fold}/{N_SPLITS}")
    print("=" * 80)

    # =========================
    # Split del fold
    # =========================
    df_train_fold = df_train.iloc[train_idx].copy()
    df_val_fold = df_train.iloc[val_idx].copy()

    print("Shape train fold original:", df_train_fold.shape)
    print("Shape val fold original  :", df_val_fold.shape)
    print()

    # =========================
    # Rebalanceo SOLO sobre train fold
    # =========================
    df_train_fold_balanceado = rebalancear_train_fold(
        df_fold_train=df_train_fold,
        label_col=LABEL_COL,
        target_n=TARGET_N,
        random_state=RANDOM_STATE + fold,
        nearmiss_version=NEARMISS_VERSION,
        smote_k_neighbors=SMOTE_K_NEIGHBORS,
        estrategia_rebalanceo=ESTRATEGIA_DE_REBALANCEO,
        enn_n_neighbors=ENN_N_NEIGHBORS
    )

    X_train_fold_bal = df_train_fold_balanceado.drop(columns=[LABEL_COL])
    y_train_fold_bal = df_train_fold_balanceado[LABEL_COL]

    X_val_fold = df_val_fold.drop(columns=[LABEL_COL])
    y_val_fold = df_val_fold[LABEL_COL]

    # =========================
    # Modelo nuevo para cada fold
    # =========================
    pipeline_fold = Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=N_COMPONENTS_PCA)),
        ("mlp", MLPClassifier(
            hidden_layer_sizes=MLP_HIDDEN_LAYER_SIZES,
            activation=MLP_ACTIVATION,
            solver=MLP_SOLVER,
            alpha=MLP_ALPHA,
            batch_size=MLP_BATCH_SIZE,
            learning_rate=MLP_LEARNING_RATE,
            learning_rate_init=MLP_LEARNING_RATE_INIT,
            max_iter=MLP_MAX_ITER,
            early_stopping=MLP_EARLY_STOPPING,
            validation_fraction=MLP_VALIDATION_FRACTION,
            n_iter_no_change=MLP_N_ITER_NO_CHANGE,
            verbose=MLP_VERBOSE,
            random_state=RANDOM_STATE + fold
        ))
    ])
    
    # =========================
    # Entrenamiento
    # =========================
    t0 = time.time()
    pipeline_fold.fit(X_train_fold_bal, y_train_fold_bal)
    fit_time = time.time() - t0

    # =========================
    # Validación
    # =========================
    t0 = time.time()
    y_pred_val = pipeline_fold.predict(X_val_fold)
    score_time = time.time() - t0

    roc_auc_val = calcular_roc_auc_multiclase_seguro(
        modelo=pipeline_fold,
        X_val=X_val_fold,
        y_val=y_val_fold,
        labels_globales=labels_globales
    )

    metricas_fold = {
        "fold": fold,

        "train_original_rows": int(df_train_fold.shape[0]),
        "train_balanceado_rows": int(df_train_fold_balanceado.shape[0]),
        "val_rows": int(df_val_fold.shape[0]),

        "accuracy": accuracy_score(y_val_fold, y_pred_val),

        "precision_weighted": precision_score(
            y_val_fold, y_pred_val, average="weighted", zero_division=0
        ),
        "recall_weighted": recall_score(
            y_val_fold, y_pred_val, average="weighted", zero_division=0
        ),
        "f1_weighted": f1_score(
            y_val_fold, y_pred_val, average="weighted", zero_division=0
        ),

        "precision_macro": precision_score(
            y_val_fold, y_pred_val, average="macro", zero_division=0
        ),
        "recall_macro": recall_score(
            y_val_fold, y_pred_val, average="macro", zero_division=0
        ),
        "f1_macro": f1_score(
            y_val_fold, y_pred_val, average="macro", zero_division=0
        ),

        "mcc": matthews_corrcoef(y_val_fold, y_pred_val),
        "roc_auc": roc_auc_val,

        "fit_time": fit_time,
        "score_time": score_time
    }

    resultados_folds.append(metricas_fold)

    print("Métricas fold:")
    print(metricas_fold)
    print()

In [ ]:
df_folds = pd.DataFrame(resultados_folds)

df_folds

In [ ]:
summary_cv = {
    "experimento": NOMBRE_EXPERIMENTO,
    "dataset_train": str(RUTA_DATASET / NOMBRE_DATASET_TRAIN),
    "shape_train": {
        "rows": int(df_train.shape[0]),
        "cols": int(df_train.shape[1])
    },
    "parametros": {
        "modelo": "MLPClassifier",
        "mlp_hidden_layer_sizes": MLP_HIDDEN_LAYER_SIZES,
        "mlp_activation": MLP_ACTIVATION,
        "mlp_solver": MLP_SOLVER,
        "mlp_alpha": MLP_ALPHA,
        "mlp_batch_size": MLP_BATCH_SIZE,
        "mlp_learning_rate": MLP_LEARNING_RATE,
        "mlp_learning_rate_init": MLP_LEARNING_RATE_INIT,
        "mlp_max_iter": MLP_MAX_ITER,
        "mlp_early_stopping": MLP_EARLY_STOPPING,
        "mlp_validation_fraction": MLP_VALIDATION_FRACTION,
        "mlp_n_iter_no_change": MLP_N_ITER_NO_CHANGE,
        "n_components_pca": N_COMPONENTS_PCA,
        "estrategia_rebalanceo": ESTRATEGIA_DE_REBALANCEO,
        "target_n": TARGET_N,
        "nearmiss_version": NEARMISS_VERSION,
        "smote_k_neighbors": SMOTE_K_NEIGHBORS,
        "enn_n_neighbors": ENN_N_NEIGHBORS
    },
    "metricas_media": {
        "accuracy": float(df_folds["accuracy"].mean()),

        "precision_weighted": float(df_folds["precision_weighted"].mean()),
        "recall_weighted": float(df_folds["recall_weighted"].mean()),
        "f1_weighted": float(df_folds["f1_weighted"].mean()),

        "precision_macro": float(df_folds["precision_macro"].mean()),
        "recall_macro": float(df_folds["recall_macro"].mean()),
        "f1_macro": float(df_folds["f1_macro"].mean()),

        "mcc": float(df_folds["mcc"].mean()),
        "roc_auc": float(df_folds["roc_auc"].mean()),
        "fit_time": float(df_folds["fit_time"].mean()),
        "score_time": float(df_folds["score_time"].mean())
    },
    "metricas_std": {
        "accuracy": float(df_folds["accuracy"].std(ddof=1)),

        "precision_weighted": float(df_folds["precision_weighted"].std(ddof=1)),
        "recall_weighted": float(df_folds["recall_weighted"].std(ddof=1)),
        "f1_weighted": float(df_folds["f1_weighted"].std(ddof=1)),

        "precision_macro": float(df_folds["precision_macro"].std(ddof=1)),
        "recall_macro": float(df_folds["recall_macro"].std(ddof=1)),
        "f1_macro": float(df_folds["f1_macro"].std(ddof=1)),

        "mcc": float(df_folds["mcc"].std(ddof=1)),
        "roc_auc": float(df_folds["roc_auc"].std(ddof=1)),
        "fit_time": float(df_folds["fit_time"].std(ddof=1)),
        "score_time": float(df_folds["score_time"].std(ddof=1))
    }
}

summary_cv

In [ ]:
print("Accuracy\tPrecision weighted\tRecall weighted\tF1 weighted\tPrecision macro\tRecall macro\tF1 macro\tMCC\tROC AUC")

print(
    f"{summary_cv['metricas_media']['accuracy']:.6f} ± {summary_cv['metricas_std']['accuracy']:.6f}\t"
    f"{summary_cv['metricas_media']['precision_weighted']:.6f} ± {summary_cv['metricas_std']['precision_weighted']:.6f}\t"
    f"{summary_cv['metricas_media']['recall_weighted']:.6f} ± {summary_cv['metricas_std']['recall_weighted']:.6f}\t"
    f"{summary_cv['metricas_media']['f1_weighted']:.6f} ± {summary_cv['metricas_std']['f1_weighted']:.6f}\t"
    f"{summary_cv['metricas_media']['precision_macro']:.6f} ± {summary_cv['metricas_std']['precision_macro']:.6f}\t"
    f"{summary_cv['metricas_media']['recall_macro']:.6f} ± {summary_cv['metricas_std']['recall_macro']:.6f}\t"
    f"{summary_cv['metricas_media']['f1_macro']:.6f} ± {summary_cv['metricas_std']['f1_macro']:.6f}\t"
    f"{summary_cv['metricas_media']['mcc']:.6f} ± {summary_cv['metricas_std']['mcc']:.6f}\t"
    f"{summary_cv['metricas_media']['roc_auc']:.6f} ± {summary_cv['metricas_std']['roc_auc']:.6f}"
)

In [ ]:
ruta_cv_csv = RUTA_RESULTADOS / NOMBRE_RESULTADOS_CV_CSV
df_folds.to_csv(ruta_cv_csv, index=False)

print("Resultados por fold guardados en:")
print(ruta_cv_csv.resolve())

In [ ]:
ruta_cv_json = RUTA_RESULTADOS / NOMBRE_RESULTADOS_CV_JSON

with open(ruta_cv_json, "w", encoding="utf-8") as f:
    json.dump(summary_cv, f, indent=4, ensure_ascii=False)

print("Resumen CV guardado en:")
print(ruta_cv_json.resolve())

In [ ]:
df_folds

In [ ]:
df_test = cargar_dataset(
    nombre_dataset=NOMBRE_DATASET_TEST,
    ruta_base=RUTA_DATASET
)

print("Forma del dataset test:")
print(df_test.shape)

df_test.head()

In [ ]:
if LABEL_COL not in df_test.columns:
    raise ValueError(f"No se encontró la columna {LABEL_COL} en test")

print("Última columna test:", df_test.columns[-1])
print("Tipo de LABEL test:", df_test[LABEL_COL].dtype)
print()

print("Distribución de clases en test:")
display(df_test[LABEL_COL].value_counts(dropna=False).to_frame("count"))

In [ ]:
X_test = df_test.drop(columns=[LABEL_COL]).copy()
y_test = df_test[LABEL_COL].copy()

print("Shape X_test:", X_test.shape)
print("Shape y_test:", y_test.shape)

In [ ]:
columnas_no_numericas_test = X_test.select_dtypes(exclude=[np.number]).columns.tolist()

print("Columnas no numéricas en X_test:")
print(columnas_no_numericas_test)

if len(columnas_no_numericas_test) > 0:
    raise ValueError("Hay columnas no numéricas en X_test. Revísalas antes de seguir.")

In [ ]:
print("Rebalanceando todo el train original para entrenar el modelo final...")

df_train_balanceado_final = rebalancear_train_fold(
    df_fold_train=df_train,
    label_col=LABEL_COL,
    target_n=TARGET_N,
    random_state=RANDOM_STATE,
    nearmiss_version=NEARMISS_VERSION,
    smote_k_neighbors=SMOTE_K_NEIGHBORS,
    estrategia_rebalanceo=ESTRATEGIA_DE_REBALANCEO,
    enn_n_neighbors=ENN_N_NEIGHBORS
)

X_train_balanceado_final = df_train_balanceado_final.drop(columns=[LABEL_COL])
y_train_balanceado_final = df_train_balanceado_final[LABEL_COL]

pipeline.fit(X_train_balanceado_final, y_train_balanceado_final)

print("Modelo final entrenado con todo el train rebalanceado.")
print("Train original   :", df_train.shape)
print("Train balanceado :", df_train_balanceado_final.shape)

In [ ]:
y_pred_test = pipeline.predict(X_test)

print("Predicciones en test generadas.")
print("Número de predicciones:", len(y_pred_test))

y_proba_test = pipeline.predict_proba(X_test)

roc_auc_test = roc_auc_score(
    y_test,
    y_proba_test,
    multi_class="ovr",
    average="weighted"
)

In [ ]:
metricas_test = {
    "accuracy": accuracy_score(y_test, y_pred_test),

    "precision_weighted": precision_score(y_test, y_pred_test, average="weighted", zero_division=0),
    "recall_weighted": recall_score(y_test, y_pred_test, average="weighted", zero_division=0),
    "f1_weighted": f1_score(y_test, y_pred_test, average="weighted", zero_division=0),

    "precision_macro": precision_score(y_test, y_pred_test, average="macro", zero_division=0),
    "recall_macro": recall_score(y_test, y_pred_test, average="macro", zero_division=0),
    "f1_macro": f1_score(y_test, y_pred_test, average="macro", zero_division=0),

    "roc_auc": roc_auc_test,

    "mcc": matthews_corrcoef(y_test, y_pred_test)
}

metricas_test

In [ ]:
print("Accuracy\tPrecision weighted\tRecall weighted\tF1 weighted\tPrecision macro\tRecall macro\tF1 macro\tMCC\tROC AUC")

print(
    f"{metricas_test['accuracy']:.6f}\t"
    f"{metricas_test['precision_weighted']:.6f}\t"
    f"{metricas_test['recall_weighted']:.6f}\t"
    f"{metricas_test['f1_weighted']:.6f}\t"
    f"{metricas_test['precision_macro']:.6f}\t"
    f"{metricas_test['recall_macro']:.6f}\t"
    f"{metricas_test['f1_macro']:.6f}\t"
    f"{metricas_test['mcc']:.6f}\t"
    f"{metricas_test['roc_auc']:.6f}"
)

In [ ]:
labels_ordenadas = sorted(pd.unique(pd.concat([y_test, pd.Series(y_pred_test)])))

cm = confusion_matrix(y_test, y_pred_test, labels=labels_ordenadas)
df_cm = pd.DataFrame(cm, index=labels_ordenadas, columns=labels_ordenadas)

print("Matriz de confusión en test:")
display(df_cm)

In [ ]:
print("========== CLASSIFICATION REPORT TEST ==========")
print(classification_report(y_test, y_pred_test, zero_division=0))

In [ ]:
summary_test = {
    "experimento": NOMBRE_EXPERIMENTO,
    "dataset_test": str(RUTA_DATASET / NOMBRE_DATASET_TEST),
    "shape_test": {
        "rows": int(df_test.shape[0]),
        "cols": int(df_test.shape[1])
    },
    "parametros": {
        "modelo": "MLPClassifier",
        "mlp_hidden_layer_sizes": MLP_HIDDEN_LAYER_SIZES,
        "mlp_activation": MLP_ACTIVATION,
        "mlp_solver": MLP_SOLVER,
        "mlp_alpha": MLP_ALPHA,
        "mlp_batch_size": MLP_BATCH_SIZE,
        "mlp_learning_rate": MLP_LEARNING_RATE,
        "mlp_learning_rate_init": MLP_LEARNING_RATE_INIT,
        "mlp_max_iter": MLP_MAX_ITER,
        "mlp_early_stopping": MLP_EARLY_STOPPING,
        "mlp_validation_fraction": MLP_VALIDATION_FRACTION,
        "mlp_n_iter_no_change": MLP_N_ITER_NO_CHANGE,
        "n_components_pca": N_COMPONENTS_PCA,
        "estrategia_rebalanceo": ESTRATEGIA_DE_REBALANCEO,
        "target_n": TARGET_N,
        "nearmiss_version": NEARMISS_VERSION,
        "smote_k_neighbors": SMOTE_K_NEIGHBORS,
        "enn_n_neighbors": ENN_N_NEIGHBORS
    },
    "metricas_test": {
        "accuracy": float(metricas_test["accuracy"]),

        "precision_weighted": float(metricas_test["precision_weighted"]),
        "recall_weighted": float(metricas_test["recall_weighted"]),
        "f1_weighted": float(metricas_test["f1_weighted"]),

        "precision_macro": float(metricas_test["precision_macro"]),
        "recall_macro": float(metricas_test["recall_macro"]),
        "f1_macro": float(metricas_test["f1_macro"]),

        "mcc": float(metricas_test["mcc"])
    }
}

summary_test

In [ ]:
df_metricas_test = pd.DataFrame([metricas_test])

ruta_test_csv = RUTA_RESULTADOS / NOMBRE_RESULTADOS_TEST_CSV
df_metricas_test.to_csv(ruta_test_csv, index=False)

print("Métricas test guardadas en:")
print(ruta_test_csv.resolve())

In [ ]:
ruta_cm_csv = RUTA_RESULTADOS / NOMBRE_RESULTADOS_TEST_CM_CSV
df_cm.to_csv(ruta_cm_csv, index=True)

print("Matriz de confusión test guardada en:")
print(ruta_cm_csv.resolve())

In [ ]:
ruta_test_json = RUTA_RESULTADOS / NOMBRE_RESULTADOS_TEST_JSON

with open(ruta_test_json, "w", encoding="utf-8") as f:
    json.dump(summary_test, f, indent=4, ensure_ascii=False)

print("Resumen test guardado en:")
print(ruta_test_json.resolve())

In [ ]:
print("========== RESUMEN FINAL ==========")
print("CV:")
print(summary_cv["metricas_media"])
print()
print("TEST:")
print(summary_test["metricas_test"])